In [ ]:
```xml
<VSCode.Cell language="markdown">
# 📊 Advanced Monitoring & Observability Guide

**Deep observability with metrics, logs, traces, and anomaly detection**

This guide covers:
- Observability foundations (metrics, logs, traces, profiles)
- Prometheus metrics collection
- Distributed tracing with Jaeger/Zipkin
- Log aggregation with ELK/Loki
- Alerting strategies
- Anomaly detection
- Service level objectives (SLOs)
- Dashboard design for operations
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Metrics & Instrumentation

### Prometheus Metrics
```python
from prometheus_client import Counter, Histogram, Gauge, start_http_server
import time

class ApplicationMetrics:
    """Instrument application with Prometheus metrics"""
    
    def __init__(self):
        # Counters (monotonically increasing)
        self.requests_total = Counter(
            'http_requests_total',
            'Total HTTP requests',
            ['method', 'endpoint', 'status']
        )
        
        self.errors_total = Counter(
            'errors_total',
            'Total errors by type',
            ['error_type']
        )
        
        # Histograms (measure distribution)
        self.request_duration = Histogram(
            'http_request_duration_seconds',
            'HTTP request duration',
            ['method', 'endpoint'],
            buckets=(0.1, 0.5, 1.0, 2.5, 5.0, 10.0)
        )
        
        # Gauges (can go up or down)
        self.active_connections = Gauge(
            'active_connections',
            'Number of active connections'
        )
        
        self.cache_size = Gauge(
            'cache_size_bytes',
            'Cache size in bytes'
        )
        
        # Start metrics server
        start_http_server(8000)
    
    def record_request(self, method: str, endpoint: str, status: int, duration: float):
        """Record HTTP request metrics"""
        self.requests_total.labels(
            method=method,
            endpoint=endpoint,
            status=status
        ).inc()
        
        self.request_duration.labels(
            method=method,
            endpoint=endpoint
        ).observe(duration)
    
    def record_error(self, error_type: str):
        """Record error"""
        self.errors_total.labels(error_type=error_type).inc()
    
    def update_connections(self, count: int):
        """Update active connections gauge"""
        self.active_connections.set(count)

# Usage
metrics = ApplicationMetrics()

# In request handler
start_time = time.time()
try:
    # Handle request
    duration = time.time() - start_time
    metrics.record_request('GET', '/api/users', 200, duration)
except Exception as e:
    metrics.record_error(type(e).__name__)
```

### Custom Metrics Collection
```python
class CustomMetricsCollector:
    """Collect business and technical metrics"""
    
    def __init__(self):
        self.metrics = {}
    
    def increment_counter(self, metric_name: str, value: float = 1, tags: dict = None):
        """Increment counter metric"""
        key = self._create_key(metric_name, tags)
        
        if key not in self.metrics:
            self.metrics[key] = {'type': 'counter', 'value': 0, 'tags': tags or {}}
        
        self.metrics[key]['value'] += value
    
    def record_gauge(self, metric_name: str, value: float, tags: dict = None):
        """Record gauge metric"""
        key = self._create_key(metric_name, tags)
        self.metrics[key] = {'type': 'gauge', 'value': value, 'tags': tags or {}}
    
    def record_histogram(self, metric_name: str, value: float, tags: dict = None):
        """Record histogram metric"""
        key = self._create_key(metric_name, tags)
        
        if key not in self.metrics:
            self.metrics[key] = {'type': 'histogram', 'values': [], 'tags': tags or {}}
        
        self.metrics[key]['values'].append(value)
    
    def _create_key(self, metric_name: str, tags: dict = None) -> str:
        """Create unique key for metric"""
        if tags:
            tag_str = ','.join(f"{k}={v}" for k, v in sorted(tags.items()))
            return f"{metric_name}[{tag_str}]"
        return metric_name
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Distributed Tracing

### Jaeger Tracing Setup
```python
from jaeger_client import Config
import logging

class DistributedTracing:
    """Setup distributed tracing with Jaeger"""
    
    @staticmethod
    def init_tracer(service_name: str):
        """Initialize Jaeger tracer"""
        config = Config(
            config={
                'sampler': {
                    'type': 'const',
                    'param': 1,  # Sample all traces
                },
                'logging': True,
            },
            service_name=service_name,
            validate=True,
        )
        
        return config.initialize_tracer()

class TracedService:
    """Service with distributed tracing"""
    
    def __init__(self, tracer):
        self.tracer = tracer
    
    def process_request(self, request_id: str):
        """Process request with tracing"""
        
        # Create root span
        with self.tracer.start_active_span('process_request') as scope:
            scope.span.set_tag('request_id', request_id)
            
            # Call service 1
            result1 = self._call_service_1()
            
            # Call service 2
            result2 = self._call_service_2(result1)
            
            return result2
    
    def _call_service_1(self):
        """Service 1 with child span"""
        with self.tracer.start_active_span('service_1_call') as scope:
            scope.span.set_tag('service', 'service1')
            return "result1"
    
    def _call_service_2(self, data):
        """Service 2 with child span"""
        with self.tracer.start_active_span('service_2_call') as scope:
            scope.span.set_tag('service', 'service2')
            return "result2"

# Usage
tracer = DistributedTracing.init_tracer('my-service')
service = TracedService(tracer)
result = service.process_request('req-123')
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Alerting & Anomaly Detection

### Rule-Based Alerting
```python
class AlertRule:
    """Alert rule definition"""
    
    def __init__(self, name: str, condition, severity: str):
        self.name = name
        self.condition = condition
        self.severity = severity
    
    def evaluate(self, metrics: dict) -> bool:
        """Evaluate if alert should fire"""
        return self.condition(metrics)

class AlertingEngine:
    """Process and fire alerts"""
    
    def __init__(self):
        self.rules = []
        self.active_alerts = set()
    
    def add_rule(self, rule: AlertRule):
        """Register alert rule"""
        self.rules.append(rule)
    
    def evaluate_all_rules(self, metrics: dict):
        """Evaluate all rules and fire alerts"""
        
        for rule in self.rules:
            alert_id = f"{rule.name}:{metrics.get('instance', 'unknown')}"
            
            if rule.evaluate(metrics):
                if alert_id not in self.active_alerts:
                    self._fire_alert(rule, metrics)
                    self.active_alerts.add(alert_id)
            else:
                if alert_id in self.active_alerts:
                    self._resolve_alert(rule)
                    self.active_alerts.discard(alert_id)
    
    def _fire_alert(self, rule: AlertRule, metrics: dict):
        """Fire alert notification"""
        print(f"🚨 ALERT: {rule.name} (Severity: {rule.severity})")
        print(f"   Metrics: {metrics}")
    
    def _resolve_alert(self, rule: AlertRule):
        """Resolve alert"""
        print(f"✅ RESOLVED: {rule.name}")

# Usage
alerting = AlertingEngine()

# Define alert rules
high_error_rate = AlertRule(
    name="High Error Rate",
    condition=lambda m: m.get('error_rate', 0) > 0.05,
    severity="critical"
)

slow_response = AlertRule(
    name="Slow Response Time",
    condition=lambda m: m.get('p99_latency', 0) > 1000,
    severity="warning"
)

alerting.add_rule(high_error_rate)
alerting.add_rule(slow_response)

# Evaluate
metrics = {'error_rate': 0.08, 'p99_latency': 1500, 'instance': 'server-1'}
alerting.evaluate_all_rules(metrics)
```

### Anomaly Detection
```python
import numpy as np
from sklearn.ensemble import IsolationForest

class AnomalyDetector:
    """Detect anomalies in metrics"""
    
    def __init__(self, contamination: float = 0.1):
        self.model = IsolationForest(contamination=contamination)
        self.is_fitted = False
    
    def fit(self, training_data: np.ndarray):
        """Fit anomaly detection model"""
        self.model.fit(training_data)
        self.is_fitted = True
    
    def detect_anomalies(self, data: np.ndarray) -> np.ndarray:
        """Detect anomalies in data"""
        if not self.is_fitted:
            raise ValueError("Model not fitted. Call fit() first.")
        
        predictions = self.model.predict(data)
        return predictions == -1  # -1 indicates anomaly
    
    def get_anomaly_score(self, data: np.ndarray) -> np.ndarray:
        """Get anomaly scores (higher = more anomalous)"""
        return -self.model.score_samples(data)

# Usage
detector = AnomalyDetector(contamination=0.05)

# Training on normal data
training_data = np.random.normal(loc=100, scale=10, size=(1000, 3))
detector.fit(training_data)

# Detect anomalies in new data
new_data = np.array([[100, 105, 102], [50, 300, 99], [102, 101, 103]])
is_anomaly = detector.detect_anomalies(new_data)
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 4. Service Level Objectives (SLOs)

### SLO Definition & Tracking
```python
class SLO:
    """Service Level Objective"""
    
    def __init__(self, name: str, target: float, window_days: int = 30):
        self.name = name
        self.target = target  # e.g., 0.99 for 99%
        self.window_days = window_days
        self.error_budget = 1 - target  # Allowed error rate
        self.events = []
    
    def record_success(self):
        """Record successful request"""
        self.events.append({'success': True, 'timestamp': time.time()})
    
    def record_failure(self):
        """Record failed request"""
        self.events.append({'success': False, 'timestamp': time.time()})
    
    def get_current_slo(self) -> float:
        """Calculate current SLO compliance"""
        if not self.events:
            return 1.0
        
        successes = sum(1 for e in self.events if e['success'])
        return successes / len(self.events)
    
    def get_error_budget_remaining(self) -> float:
        """Calculate remaining error budget"""
        current_slo = self.get_current_slo()
        error_rate = 1 - current_slo
        
        if error_rate > self.error_budget:
            return 0.0  # Budget exhausted
        
        return self.error_budget - error_rate
    
    def is_on_track(self) -> bool:
        """Check if SLO is on track"""
        return self.get_current_slo() >= self.target

# Usage
slo = SLO(name="API Availability", target=0.99)

# Record events
for i in range(1000):
    if i % 10 == 0:  # 1% failure rate
        slo.record_failure()
    else:
        slo.record_success()

print(f"Current SLO: {slo.get_current_slo():.2%}")
print(f"Error Budget Remaining: {slo.get_error_budget_remaining():.4%}")
print(f"On Track: {slo.is_on_track()}")
```

### Error Budget Management
```python
class ErrorBudgetManager:
    """Manage error budgets across services"""
    
    def __init__(self):
        self.slos = {}
    
    def register_slo(self, service_name: str, slo: SLO):
        """Register SLO for service"""
        self.slos[service_name] = slo
    
    def can_deploy(self) -> bool:
        """Check if deployments allowed"""
        # No deployments if error budget exhausted
        for slo in self.slos.values():
            if slo.get_error_budget_remaining() <= 0:
                return False
        return True
    
    def get_budget_summary(self) -> dict:
        """Get error budget summary"""
        summary = {}
        for service_name, slo in self.slos.items():
            summary[service_name] = {
                'target': slo.target,
                'current': slo.get_current_slo(),
                'budget_remaining': slo.get_error_budget_remaining(),
                'on_track': slo.is_on_track()
            }
        return summary
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 5. Observability Checklist

✅ **Metrics**
- [ ] Key metrics identified
- [ ] Prometheus configured
- [ ] Custom metrics implemented
- [ ] Retention policy set
- [ ] Cardinality managed

✅ **Logging**
- [ ] Centralized logging set up
- [ ] Log levels configured
- [ ] Structured logging implemented
- [ ] Log retention defined
- [ ] Performance impact minimal

✅ **Tracing**
- [ ] Distributed tracing enabled
- [ ] Sampling strategy defined
- [ ] Trace sampling rate optimized
- [ ] Backend configured (Jaeger/Zipkin)
- [ ] Latency overhead acceptable

✅ **Alerting**
- [ ] Alert rules defined
- [ ] Alert channels configured (email, Slack, PagerDuty)
- [ ] Alert thresholds tuned
- [ ] False positive rate minimized
- [ ] Run books documented

✅ **SLOs**
- [ ] SLOs defined for each service
- [ ] Error budgets calculated
- [ ] Tracking implemented
- [ ] Dashboard displays SLO status
- [ ] Deployment gates configured
</MySCode.Cell>
```